# 测试集性能评估 (Test Set Evaluation)

本模块用于在完全独立且未参与模型训练的**测试集**上，评估已训练完成的 UNet 模型的最终性能，输出涵盖各类别交并比（IoU）、准确率（Acc）等指标的综合测试报告。

## 1. 环境准备与工作目录锁定

锁定绝对工作路径至 `mmsegmentation` 根目录，并主动释放由于之前训练任务遗留在 GPU 中的显存。确保能够正确调用官方测试脚本。

In [1]:
import os
import glob
import torch

# 1. 锁定绝对工作目录
WORK_DIR = '/root/LearningMMSegmentation/mmsegmentation'
os.chdir(WORK_DIR)

# 2. 强制清理 GPU 显存残留，防止 CUDA out of memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 GPU 显存已强制清理。")

print(f"✅ 环境初始化完成。当前工作目录: {os.getcwd()}")

🧹 GPU 显存已强制清理。
✅ 环境初始化完成。当前工作目录: /root/LearningMMSegmentation/mmsegmentation


## 2. 自动定位配置文件与最优模型权重

在 `F1.` 的配置中，已设定 `save_best='mIoU'`。MMEngine 在训练过程中会自动保存验证集表现最佳的模型权重（如 `best_mIoU_iter_xxxx.pth`）。本节利用路径扫描技术自动提取该权重文件，消除手动配置的误差风险。

In [2]:
# 1. 设定训练阶段导出的自定义配置文件路径
config_file = 'My-Configs/CustomDataset_UNet_pipeline.py'

# 2. 设定训练结果的输出目录
checkpoint_dir = 'work_dirs/CustomDataset_UNet_pipeline'

# 3. 自动扫描以 'best_mIoU' 开头的权重文件
best_checkpoints = glob.glob(os.path.join(checkpoint_dir, 'best_mIoU*.pth'))

if len(best_checkpoints) == 0:
    print(f"❌ 错误：未在 {checkpoint_dir} 目录下找到最优模型权重！")
    checkpoint_path = None
else:
    # 提取列表中的最后一个（即最新的）最优权重
    checkpoint_path = best_checkpoints[-1]
    print(f"✅ 成功定位最优模型权重:\n👉 {checkpoint_path}")

✅ 成功定位最优模型权重:
👉 work_dirs/CustomDataset_UNet_pipeline/best_mIoU_iter_1750.pth


## 3. 执行测试集正向推理与性能评估

**运行前先清理内核**：在 Jupyter 侧边栏清理。把**正在运行终端和内核**全部关闭。
调用 MMSegmentation 提供的原生 `tools/test.py` 脚本，载入配置文件与最优权重，对测试集流水线（Test Pipeline）中的数据进行逐一推理，并输出包含详细类别指标的终端报告。

In [3]:
if checkpoint_path is not None and os.path.exists(config_file):
    print("🚀 开始在测试集上执行全量性能评估...")
    print("-" * 60)
    
    # 采用 %run 魔法命令在当前内核环境变量下执行测试脚本
    %run tools/test.py {config_file} {checkpoint_path}
    
    print("-" * 60)
    print("🎉 测试集性能评估执行完毕！请查看上方终端输出的各类别成绩单。")
else:
    print("⚠️ 前置条件未满足（配置文件或权重文件缺失），无法启动测试。")

🚀 开始在测试集上执行全量性能评估...
------------------------------------------------------------
04/03 18:04:45 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.7.16 (default, Jan 17 2023, 22:20:44) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 0
    GPU 0: NVIDIA GeForce RTX 3090
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 11.3, V11.3.109
    GCC: gcc (Ubuntu 9.3.0-17ubuntu1~20.04) 9.3.0
    PyTorch: 1.10.1+cu113
    PyTorch compiling details: PyTorch built with:
  - GCC 7.3
  - C++ Version: 201402
  - Intel(R) Math Kernel Library Version 2020.0.0 Product Build 20191122 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.2.3 (Git Hash 7336ca9f055cf1bfa13efb658fe15dc9b41f0740)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runt

/root/LearningMMSegmentation/mmsegmentation/mmseg/models/builder.py:36: UserWarning: ``build_loss`` would be deprecated soon, please use ``mmseg.registry.MODELS.build()`` 
  warnings.warn('``build_loss`` would be deprecated soon, please use '
/root/LearningMMSegmentation/mmsegmentation/mmseg/models/losses/cross_entropy_loss.py:236: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  'Default ``avg_non_ignore`` is False, if you would like to '


04/03 18:04:53 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
04/03 18:04:53 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                

/root/LearningMMSegmentation/mmsegmentation/mmseg/engine/hooks/visualization_hook.py:61: UserWarning: The draw is False, it means that the hook for visualization will not take effect. The results will NOT be visualized or stored.
  warnings.warn('The draw is False, it means that the '


04/03 18:04:53 - mmengine - WARNING - The prefix is not set in metric class IoUMetric.
Loads checkpoint by local backend from path: work_dirs/CustomDataset_UNet_pipeline/best_mIoU_iter_1750.pth
04/03 18:04:54 - mmengine - INFO - Load checkpoint from work_dirs/CustomDataset_UNet_pipeline/best_mIoU_iter_1750.pth
04/03 18:05:07 - mmengine - INFO - per class results:
04/03 18:05:07 - mmengine - INFO - 
+------------+-------+-------+-------+--------+-----------+--------+
|   Class    |  IoU  |  Acc  |  Dice | Fscore | Precision | Recall |
+------------+-------+-------+-------+--------+-----------+--------+
| background | 90.02 | 94.46 | 94.75 | 94.75  |   95.04   | 94.46  |
|    red     |  88.6 | 98.31 | 93.95 | 93.95  |   89.97   | 98.31  |
|   green    | 59.51 | 66.14 | 74.62 | 74.62  |   85.59   | 66.14  |
|   white    | 74.28 | 82.71 | 85.24 | 85.24  |   87.94   | 82.71  |
| seed-black | 62.35 | 79.59 | 76.81 | 76.81  |   74.21   | 79.59  |
| seed-white |  0.0  |  0.0  |  0.0  |  nan   

## 特别注意，在同一个benchmark上面对比性能指标，才有意义